# Project 2: COVID-19 Global Impact Dashboard
**VEDA Technology | Data Analyst Internship**

## Business question
How did reported COVID-19 cases, deaths, and vaccination progress vary across countries and over time?

**Important:** The notebook uses the source dataset when run. Do not copy numerical findings into the report until the notebook has completed successfully and you have reviewed the generated tables and charts.

## 1. Setup and imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
# Supports running from either project root or notebooks/ directory
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis import load_source, prepare_data, latest_country_summary, quality_report

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
OUT_DIR = PROJECT_ROOT / "outputs"
for folder in [RAW_DIR, CLEAN_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Project folder:", PROJECT_ROOT.resolve())

## 2. Load data

In [ ]:
DATA_URL = "https://covid.ourworldindata.org/data/owid-covid-data.csv"
raw_path = RAW_DIR / "owid-covid-data.csv"

if raw_path.exists():
    raw = pd.read_csv(raw_path, low_memory=False)
    print("Loaded existing raw file:", raw_path)
else:
    print("Downloading OWID data. This requires an internet connection and may take a few minutes.")
    raw = pd.read_csv(DATA_URL, low_memory=False)
    raw.to_csv(raw_path, index=False)
    print("Saved raw source to:", raw_path)

print("Raw shape:", raw.shape)
display(raw.head())
print("Columns:", raw.columns.tolist()[:25])

## 3. Initial data-quality inspection

In [ ]:
print("Duplicate full rows:", int(raw.duplicated().sum()))
print("Missing values in top 15 columns:")
display(raw.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))
if "date" in raw:
    print("Date range:", raw["date"].min(), "to", raw["date"].max())
if "location" in raw:
    print("Unique locations:", raw["location"].nunique())

## 4. Cleaning and feature engineering

In [ ]:
clean = prepare_data(raw)
clean_path = CLEAN_DIR / "covid19_cleaned_analysis.csv"
clean.to_csv(clean_path, index=False)
print("Cleaned shape:", clean.shape)
print("Saved cleaned data:", clean_path)
display(clean.head())

### Cleaning decisions

- Kept the fields required for the project instead of carrying every source column into analysis.
- Converted `date` to datetime and numeric metrics to numeric types.
- Removed duplicate `location` + `date` observations, keeping the last record.
- Sorted each location's time series before calculating rolling averages.
- Preserved missing values instead of filling all blanks with zero.
- Added 7-day rolling averages and a cumulative case-fatality ratio (CFR).

## 5. Data quality report

In [ ]:
quality = quality_report(raw, clean)
quality_df = pd.DataFrame([quality]).T.rename(columns={0: "value"})
display(quality_df)
quality_df.to_csv(OUT_DIR / "data_quality_summary.csv")

## 6. Select countries and date window

In [ ]:
# Change this list to compare other countries.
preferred_countries = ["United States", "India", "Brazil", "United Kingdom", "France"]
available = set(clean["location"].dropna().unique())
countries = [c for c in preferred_countries if c in available]
if not countries:
    countries = clean.loc[clean["continent"].notna(), "location"].dropna().unique()[:5].tolist()
print("Countries selected:", countries)

country_data = clean[clean["location"].isin(countries)].copy()
# Restrict to rows with dates; full historical data remains in cleaned CSV.
display(country_data[["location", "date", "new_cases", "new_deaths", "people_fully_vaccinated_per_hundred"]].tail(10))

## 7. Trend analysis — daily cases

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for country in countries:
    d = country_data[country_data["location"] == country]
    ax.plot(d["date"], d["new_cases_7day_avg"], label=country, linewidth=1.8)
ax.set_title("COVID-19 daily new cases — 7-day rolling average")
ax.set_xlabel("Date"); ax.set_ylabel("Reported cases (7-day average)")
ax.legend(ncol=2); fig.tight_layout()
fig.savefig(OUT_DIR / "daily_cases_7day_average.png", dpi=160)
plt.show()

## 8. Trend analysis — daily deaths

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for country in countries:
    d = country_data[country_data["location"] == country]
    ax.plot(d["date"], d["new_deaths_7day_avg"], label=country, linewidth=1.8)
ax.set_title("COVID-19 daily new deaths — 7-day rolling average")
ax.set_xlabel("Date"); ax.set_ylabel("Reported deaths (7-day average)")
ax.legend(ncol=2); fig.tight_layout()
fig.savefig(OUT_DIR / "daily_deaths_7day_average.png", dpi=160)
plt.show()

## 9. Latest reported country comparison

In [ ]:
summary = latest_country_summary(clean)
# Use sovereign countries only where continent is available; exclude aggregate rows.
if "continent" in clean.columns:
    country_names = set(clean.loc[clean["continent"].notna(), "location"].unique())
    summary = summary[summary["location"].isin(country_names)]
summary = summary.sort_values("total_cases_per_million", ascending=False, na_position="last")
display(summary.head(15))
summary.to_csv(OUT_DIR / "latest_country_summary.csv", index=False)

## 10. Vaccination rollout

In [ ]:
vax_col = "people_fully_vaccinated_per_hundred"
if vax_col in country_data.columns and country_data[vax_col].notna().any():
    fig, ax = plt.subplots(figsize=(12, 6))
    for country in countries:
        d = country_data[country_data["location"] == country]
        ax.plot(d["date"], d[vax_col], label=country, linewidth=1.8)
    ax.set_title("People fully vaccinated (% of population, as reported)")
    ax.set_xlabel("Date"); ax.set_ylabel("Percent of population")
    ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(OUT_DIR / "vaccination_progress.png", dpi=160)
    plt.show()
else:
    print("Vaccination coverage is unavailable in this data extract; verify source columns and date coverage.")

## 11. Derived KPIs

In [ ]:
# The most recent non-null observation for each selected country and metric.
kpi_rows = []
for country in countries:
    d = clean[clean["location"] == country].sort_values("date")
    row = {"location": country, "last_observation_date": d["date"].max()}
    for col in ["total_cases", "total_deaths", "total_cases_per_million",
                "total_deaths_per_million", "people_vaccinated_per_hundred",
                "people_fully_vaccinated_per_hundred", "case_fatality_ratio_pct"]:
        if col in d.columns:
            vals = d.loc[d[col].notna(), col]
            row[col] = vals.iloc[-1] if len(vals) else np.nan
    kpi_rows.append(row)
kpis = pd.DataFrame(kpi_rows)
display(kpis)
kpis.to_csv(OUT_DIR / "selected_country_kpis.csv", index=False)

## 12. Focused insight prompts

Use the computed outputs to answer these questions in your own words:
1. Which selected country has the highest reported cases per million in the latest available observations?
2. When are the major waves visible in the 7-day case averages? Are reporting gaps or spikes present?
3. How does the timing of vaccination rollout compare across the selected countries?
4. Why can a high reported case-fatality ratio not be interpreted as a direct measure of individual risk?
5. What limitations affect fair country-to-country comparisons?

Do not claim vaccination caused lower deaths based only on these descriptive charts. Confounding factors, age structure, reporting practices, and time lags matter.

## 13. Export final files

In [ ]:
# A concise text output for report writing; values are calculated from the loaded dataset.
with open(OUT_DIR / "analysis_notes.txt", "w", encoding="utf-8") as f:
    f.write("COVID-19 Global Impact Dashboard — generated analysis notes\n")
    f.write(f"Dataset rows after cleaning: {len(clean):,}\n")
    f.write(f"Locations after cleaning: {clean['location'].nunique():,}\n")
    f.write(f"Date range: {clean['date'].min().date()} to {clean['date'].max().date()}\n")
    f.write(f"Selected countries: {', '.join(countries)}\n")
    f.write("\nQuality summary:\n")
    for k, v in quality.items():
        f.write(f"- {k}: {v}\n")
print("Exports written to:", OUT_DIR)
print("Before submission, inspect every chart and write only insights supported by these outputs.")